In [1]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime
from skopt.space import Real
from skopt import gp_minimize
from scipy.optimize import minimize

In [2]:
#Constants for dates

start_dl = "2023-01-01"
end_dl = "2026-11-25"

# yfinance treats end as exclusive, so download one extra day to include end_dl's close
end_download = (pd.Timestamp(end_dl) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

In [3]:
# INITIAL PORTFOLIO FILTER
# 1. Must be a company stock listed in the US or Canada
# 2. Must trade in USD or CAD
# 3. Average volume must be at least 5000
# 4. Exclude months with fewer than 18 trading days

# Read the tickers
try:
    tickers = pd.read_csv("Tickers_Example.csv", header=None)
    tickers = tickers.dropna()
except Exception as e:
    raise ValueError("Could not read ticker file") from e

eligible = []
eligible_currency = []
checked = []  # NEW: track tickers already checked
stock_info = {}  # NEW: save stock information for later

for i in range(len(tickers)):
    # NEW: remove spaces and make tickers uppercase
    symbol = str(tickers.iloc[i, 0]).strip().upper()

    # NEW: skip empty or duplicate tickers
    if symbol == "" or symbol in checked:
        continue

    checked.append(symbol)
    valid = True

    try:
        stock = yf.Ticker(symbol)
        info = stock.info

        # Company stock?
        if info.get("quoteType") != "EQUITY":
            valid = False

        # Listed in US/Canada?
        if info.get("market") not in ["us_market", "ca_market"]:
            valid = False

        # Currency in USD/CAD?
        if info.get("currency") not in ["USD", "CAD"]:
            valid = False

        # Check average trading volume
        if valid:
            hist = stock.history(
                start=start_dl,
                end=end_download
            )

            # NEW: check if volume data is missing
            if hist.empty or "Volume" not in hist.columns:
                valid = False
            else:
                dates = pd.to_datetime(hist.index)
                volume = hist["Volume"].replace(
                    [np.inf, -np.inf], np.nan
                )

                # NEW: remove invalid volume values
                volume = volume.where(volume >= 0)

                days_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).count()

                volume_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).sum()

                total_days = 0
                total_volume = 0

                for m in range(len(days_per_month)):
                    if days_per_month.iloc[m] >= 18:
                        total_days += days_per_month.iloc[m]
                        total_volume += volume_per_month.iloc[m]

                if total_days == 0:
                    valid = False
                elif total_volume / total_days < 5000:
                    valid = False

    # NEW: handle failed ticker requests
    except Exception as e:
        print(symbol, "could not be checked:", e)
        valid = False

    if valid:
        eligible.append(symbol)
        eligible_currency.append(info.get("currency"))
        stock_info[symbol] = info


# Download historical prices
stock_data = {}


if len(eligible) > 0:
    try:
        all_stock = yf.download(
            eligible,
            start=start_dl,
            end=end_download,
            group_by="ticker",
            auto_adjust=False,
            progress=False
        )

        for ticker in eligible:
            try:
                # NEW: handle single and multiple stock downloads
                if isinstance(all_stock.columns, pd.MultiIndex):
                    if ticker in all_stock.columns.get_level_values(0):
                        indiv_stock = all_stock[ticker].copy()
                    else:
                        indiv_stock = all_stock.xs(
                            ticker, axis=1, level=1
                        ).copy()
                else:
                    indiv_stock = all_stock.copy()

                required = ["High", "Low", "Close"]

                # NEW: check if any price columns are missing
                if not all(col in indiv_stock.columns for col in required):
                    print(ticker, "has missing price columns")
                    continue

                # NEW: remove rows with missing or invalid prices
                indiv_stock = indiv_stock.dropna(subset=required)

                for column in required:
                    indiv_stock = indiv_stock[
                        np.isfinite(indiv_stock[column]) &
                        (indiv_stock[column] > 0)
                    ]

                # NEW: check if enough historical data is available
                if len(indiv_stock) < 252:
                    print(ticker, "does not have enough price data")
                    continue

                # NEW: check if the stock has recent prices
                last_date = pd.Timestamp(indiv_stock.index[-1])
                last_date = last_date.tz_localize(None)

                check_date = pd.Timestamp(end_dl)

                if (check_date - last_date).days > 7:
                    print(ticker, "has outdated price data")
                    continue

                indiv_stock.index = indiv_stock.index.strftime("%Y-%m-%d")
                stock_data[ticker] = indiv_stock

            # NEW: skip stocks with failed downloads
            except (KeyError, TypeError, ValueError) as e:
                print(ticker, "could not be downloaded:", e)

    # NEW: handle failed batch downloads
    except Exception as e:
        print("Historical download failed:", e)

# Convert USD prices to CAD
cadusd = pd.Series(dtype=float)

if len(stock_data) > 0:
    try:
        cadusd = yf.Ticker("CADUSD=X").history(
            start=start_dl,
            end=end_download
        )["Close"]

        cadusd.index = cadusd.index.strftime("%Y-%m-%d")

        # NEW: remove invalid exchange rates
        cadusd = cadusd.replace([np.inf, -np.inf, 0], np.nan)
        cadusd = cadusd.dropna()

    except Exception as e:
        print("Exchange rates could not be downloaded:", e)


# Convert the remaining stocks
for ticker in list(stock_data):
    table = stock_data[ticker]
    i = eligible.index(ticker)

    if eligible_currency[i] == "USD":
        # NEW: match exchange rates to stock dates
        dates = cadusd.index.union(table.index)
        rates = cadusd.reindex(dates).sort_index().ffill()
        table["CADUSD"] = rates.reindex(table.index)

        table["Close_CAD"] = table["Close"] / table["CADUSD"]
        table["High_CAD"] = table["High"] / table["CADUSD"]
        table["Low_CAD"] = table["Low"] / table["CADUSD"]

    else:
        table["Close_CAD"] = table["Close"]
        table["High_CAD"] = table["High"]
        table["Low_CAD"] = table["Low"]

    required = ["Close_CAD", "High_CAD", "Low_CAD"]

    # NEW: remove invalid CAD prices
    table[required] = table[required].replace(
        [np.inf, -np.inf], np.nan
    )

    table = table.dropna(subset=required)

    for column in required:
        table = table[table[column] > 0]

    # NEW: check that enough valid CAD prices remain
    if len(table) < 252:
        print(ticker, "does not have enough valid CAD prices")
        del stock_data[ticker]
        continue

    # NEW: check for recent CAD prices
    last_date = pd.Timestamp(table.index[-1])

    if (check_date - last_date).days > 7:
        print(ticker, "has outdated CAD prices")
        del stock_data[ticker]
        continue

    stock_data[ticker] = table


# NEW: update eligible lists after removing stocks
final_eligible = []
final_currency = []

for i in range(len(eligible)):
    if eligible[i] in stock_data:
        final_eligible.append(eligible[i])
        final_currency.append(eligible_currency[i])

eligible = final_eligible
eligible_currency = final_currency

print("Eligible tickers:", eligible)
print("Number of eligible stocks:", len(eligible))

Eligible tickers: ['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
Number of eligible stocks: 35


## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** XX  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [4]:
def pull_market_cap(ticker, ticker_df, target_date):

    close_price = ticker_df.loc[:target_date].iloc[-1]["Close_CAD"]

    target_dt = pd.to_datetime(target_date)
    end_dt = target_dt + pd.Timedelta(days=5)
    shares_history = ticker.get_shares_full(start=target_dt.strftime("%Y-%m-%d"),
                                            end=end_dt.strftime("%Y-%m-%d"))

    if shares_history is not None and len(shares_history) > 0:
        share_count = shares_history.iloc[0]
    else:
        share_count = ticker.info.get("sharesOutstanding")

    if share_count is None:
        return np.nan
    return close_price * share_count


In [5]:
def calculate_adx(data, period=14):

    if len(data) < 2 * period:
        return pd.DataFrame(np.nan, index=data.index, columns=["adx", "DX", "+DI", "-DI", "smoothed TR"])

    data = data[["High_CAD", "Low_CAD", "Close_CAD"]].copy()

    # 1. True Range
    high_low = data["High_CAD"] - data["Low_CAD"]
    high_close = abs(data["High_CAD"] - data["Close_CAD"].shift(1))
    low_close = abs(data["Low_CAD"] - data["Close_CAD"].shift(1))

    data["Trading Range"] = high_low
    data["up gap"] = high_close
    data["down gap"] = low_close

    data["True Range"] = pd.concat(
        [high_low, high_close, low_close], axis=1
    ).max(axis=1)

    # 2. Directional Momentum
    up_move = data["High_CAD"].diff()
    down_move = -data["Low_CAD"].diff()

    pos_cond = (up_move > down_move) & (up_move > 0)
    neg_cond = (down_move > up_move) & (down_move > 0)

    data["+DM"] = np.where(pos_cond, up_move, 0)
    data["-DM"] = np.where(neg_cond, down_move, 0)

    # 3. Wilder Smoothing
    def wilder_smoothing(series, smoothing_period):
        smoothed = {}
        initial = sum(series.values[1:smoothing_period + 1])
        smoothed[smoothing_period] = initial

        for i in range(smoothing_period + 1, len(series)):
            smoothed[i] = (
                smoothed[i - 1]
                - (smoothed[i - 1] / smoothing_period)
                + series.values[i])

        return pd.Series(list(smoothed.values()), index=series.index[smoothing_period:])

    data["smoothed +DM"] = wilder_smoothing(data["+DM"], period)
    data["smoothed -DM"] = wilder_smoothing(data["-DM"], period)
    data["smoothed TR"] = wilder_smoothing(data["True Range"], period)

    # 4. Directional Index
    data["+DI"] = (data["smoothed +DM"] / data["smoothed TR"]) * 100
    data["-DI"] = (data["smoothed -DM"] / data["smoothed TR"]) * 100

    data["DX"] = (100 * abs(data["+DI"] - data["-DI"])) / (abs(data["+DI"]) + abs(data["-DI"]))

    # 5. Average Directional Index
    adx_values = {}
    initial = data["DX"].iloc[period:2 * period].mean()

    adx_values[2 * period - 1] = initial

    for i in range(2 * period, len(data)):
        adx_values[i] = ((period - 1) * adx_values[i - 1] + data["DX"].iloc[i]) / period

    data["adx"] = pd.Series(list(adx_values.values()), index=data.index[2 * period - 1:])

    data = data[["adx", "DX", "+DI", "-DI", "smoothed TR"]]

    return data

In [6]:
def rolling_price_std(data, period):

    rolling_std = data["Close_CAD"].pct_change(fill_method=None).rolling(period).std()

    return rolling_std

In [7]:
def pull_beta(ticker):

    info = ticker.info
    beta = info.get("beta")

    return beta

In [8]:
ADX_PERIODS = {"1M": 21, "3M": 63, "6M": 126}

def indicator_application(ticker, ticker_df, start_date):
    ticker = yf.Ticker(ticker)

    ticker_df["Market Cap"] = pull_market_cap(ticker, ticker_df,start_date)

    ticker_df["Rolling STD 63d"] = rolling_price_std(ticker_df, 63)

    ticker_df["Beta"] = pull_beta(ticker)

    for label, period in ADX_PERIODS.items():
        adx_df = calculate_adx(ticker_df, period=period)
        adx_df = adx_df.add_suffix(f" {label}")
        
        adx_df[f"DI spread {label}"] = (adx_df[f"+DI {label}"] - adx_df[f"-DI {label}"])
        ticker_df = ticker_df.drop(columns=adx_df.columns, errors="ignore")
        ticker_df = pd.concat([ticker_df, adx_df], axis=1)

    return ticker_df

In [9]:
# Pulls the return on equity for a stock
def pull_roe(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)

    info = ticker.info
    roe = info.get("returnOnEquity")

    if roe != None:
        roe = roe * 100

    return roe


In [10]:
# Pulls trailing and forward EPS for a stock
def pull_eps(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)

    info = ticker.info

    trailing_eps = info.get("trailingEps")
    forward_eps = info.get("forwardEps")

    return trailing_eps, forward_eps




In [11]:
# Calculates the correlation between stocks
def calculate_correlations(prices):
    returns = prices.pct_change(fill_method=None)
    return returns.corr()


# Calculates the average correlation with selected stocks
def average_correlation(candidate, selected_stocks, correlations):
    return correlations.loc[candidate, selected_stocks].mean()



In [12]:
for ticker, ticker_data in stock_data.items():
    stock_data[ticker] = indicator_application(ticker, ticker_data, end_dl)

In [13]:
# Calculates the total portfolio weight in each sector
def calculate_sector_weights(portfolio):
    sector_weights = portfolio.groupby("Sector")["Weight"].sum()
    return sector_weights


# Checks if any sector exceeds the 40% limit we have
def check_sector_limit(portfolio):
    sector_weights = calculate_sector_weights(portfolio)
    return (sector_weights <= 0.40 + 0.0000000001).all()




## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [14]:
# basically the length of our competition 
HOLD_DAYS = 6 

series_by_ticker = {}
for ticker, ticker_data in stock_data.items():
  series_by_ticker[ticker] = ticker_data["Close_CAD"]

close = pd.DataFrame(series_by_ticker)

returns = close.pct_change(fill_method=None)

fwd_returns = close.shift(-HOLD_DAYS) / close - 1

print(f"Close Table Shape: {close.shape}")

Close Table Shape: (999, 35)


In [15]:
HIGH_RET_RULES = {
    #"adx 6M": ">",
    #"DI spread 6M": ">",
    "adx 3M": ">",
    "DI spread 3M": ">",
    "adx 1M": ">",
    "DI spread 1M": ">",
    "Beta": ">",
    "Market Cap": "<",
}

In [16]:
# makes a seperate table for each indicator, with the stocks as columns
def build_tables(rules):

    #essentially every indicator in rules gets its own column 

    tables = {} 
    for col in rules: 
        col_data = {}
        for t, df in stock_data.items():
            col_data[t] = df[col]
        tables[col] = pd.DataFrame(col_data)
    return tables

high_return_tables = build_tables(HIGH_RET_RULES)

In [17]:
# picks usable historical dates and splits them into tuning and validation weeks
first_table = high_return_tables["adx 3M"]

has_indicators = first_table.notna().any(axis=1)
has_future_return = fwd_returns.notna().any(axis=1)

has_enough_prices = close.notna().sum(axis=1) >= 10

usable = first_table.index[
    has_indicators & has_future_return & has_enough_prices
]
test_dates = usable[::-HOLD_DAYS][:64][::-1]

split = int(len(test_dates) * 0.7)
tune_dates = test_dates[:split]
validation_dates = test_dates[split:]

print(len(tune_dates), "tuning weeks:    ", tune_dates[0], "->", tune_dates[-1])
print(len(validation_dates), "validation weeks:", validation_dates[0], "->", validation_dates[-1])

44 tuning weeks:     2025-05-16 -> 2026-05-28
20 validation weeks: 2026-06-05 -> 2026-11-17


In [18]:
# function that checks which stocks pass all the indicator cutoffs on a given date
def passes(tables, rules, cutoffs, date):
    mask = pd.Series(True, index=close.columns)

    for (col, direction), cutoff in zip(rules.items(), cutoffs):
        values = tables[col].loc[date]

        if direction == ">":
            this_rule = values > cutoff
        else:
            this_rule = values < cutoff

        mask = mask & this_rule 
    
    return mask 

# function that sets the range of cutoffs the Gaussian Process can test for each indicator
def make_space(tables, rules, dates):
    space = []
    for col, direction in rules.items():

        if direction == ">":
            q = [0.05, 0.90] 
        else:
            q = [0.10, 0.95]

        lo, hi = tables[col].loc[dates].stack().quantile(q)

        space.append(Real(float(lo), float(hi), name=col))

    return space


hr_space = make_space(high_return_tables, HIGH_RET_RULES, tune_dates)

In [19]:
# Calculate a growth score using ADX and DI spread
def growth_scores(picks, date):
    scores = {}

    for ticker in picks:
        adx = high_return_tables["adx 3M"].loc[date, ticker]
        di = high_return_tables["DI spread 3M"].loc[date, ticker]

        # Only reward positive directional movement
        if pd.isna(adx) or pd.isna(di) or di <= 0:
            score = 0
        else:
            score = adx * di

        scores[ticker] = score

    return pd.Series(scores)


In [20]:
# splits the weight 50/50 between equal weighting and growth scores
def growth_weights(scores):
    # Give every stock an equal base weight
    equal_weights = pd.Series(1 / len(scores), index=scores.index)

    # Give stronger stocks a larger weight
    if scores.sum() > 0:
        score_weights = scores / scores.sum()
    else:
        score_weights = equal_weights

    # Combine equal weighting and growth-based weighting
    weights = 0.5 * equal_weights + 0.5 * score_weights

    return weights



In [21]:
# function that caps the stocks weight at 15% max per stock
def cap_stock_weights(weights):
    weights = weights.copy()

    # Reduce any stock above 15%
    while weights.max() > 0.15:
        excess = 0

        for ticker in weights.index:
            if weights[ticker] > 0.15:
                excess += weights[ticker] - 0.15
                weights[ticker] = 0.15

        # Give the extra weight to stocks below the limit
        available = weights[weights < 0.15]

        if len(available) == 0:
            break

        weights.loc[available.index] += excess * available / available.sum()

    return weights


# Save the sector for each eligible stock
stock_sectors = {}

for ticker in stock_data:
    stock_sectors[ticker] = yf.Ticker(ticker).info.get("sector")
    

In [22]:
# function that caps the sectors weight at 40% per sector and makes sure it doesn't go over
def cap_sector_weights(weights, stock_sectors):
    weights = weights.copy()

    for _ in range(100):
        sector_totals = {}

        for ticker in weights.index:
            sector = stock_sectors[ticker]
            sector_totals[sector] = sector_totals.get(sector, 0) + weights[ticker]

        # Find sectors above 40%
        over = [s for s in sector_totals if sector_totals[s] > 0.40 + 1e-9]

        if not over:
            return weights

        sector = over[0]
        excess = sector_totals[sector] - 0.40

        # Reduce weights within the overweight sector
        for ticker in weights.index:
            if stock_sectors[ticker] == sector:
                weights[ticker] *= 0.40 / sector_totals[sector]

        # Find stocks in other sectors with room for more weight
        available = [
            t for t in weights.index
            if stock_sectors[t] != sector
            and weights[t] < 0.15
            and sector_totals[stock_sectors[t]] < 0.40
        ]

        if not available:
            raise ValueError("Not enough room to redistribute sector weights")

        # Redistribute excess gradually without exceeding either limit
        while excess > 1e-9:
            room = {
                t: min(0.15 - weights[t],
                       0.40 - sector_totals[stock_sectors[t]])
                for t in available
            }
            room = {t: r for t, r in room.items() if r > 1e-9}

            if not room:
                raise ValueError("Not enough room to redistribute sector weights")

            ticker = max(room, key=room.get)
            amount = min(excess, room[ticker])
            weights[ticker] += amount
            sector_totals[stock_sectors[ticker]] += amount
            excess -= amount

    raise ValueError("Could not satisfy sector limits")


In [23]:
# calculates the final growth weights and make sure they follow the sector and stock limits
def final_growth_weights(picks, date):
    scores = growth_scores(picks, date)
    weights = growth_weights(scores)

    weights = cap_stock_weights(weights)
    weights = cap_sector_weights(weights, stock_sectors)

    return weights



In [24]:
# Selects the top growing stocks and adds backup if fewer than 10 pass the filters
def select_growth_stocks(date, passed):
    adx = high_return_tables["adx 3M"].loc[date]

    # Rank stocks that passed filters first
    passed_stocks = list(adx[passed].dropna().sort_values(ascending=False).index)

    # Remaining stocks are backups
    backup_stocks = list(adx[~passed].dropna().sort_values(ascending=False).index)

    selected = passed_stocks[:20]

    # Add backup stocks if fewer than 10 were selected
    if len(selected) < 10:
        for ticker in backup_stocks:
            if len(selected) >= 10:
                break

            selected.append(ticker)

    return selected


In [25]:
MIN_TICKERS = 10   #setting the punishment for min tickers above the min of 10. I'd rather have more than less. 
MAX_TICKERS = 20   # max 

# tests our stock selection and weighting strategy based on historical dates
def weekly_high_return(cutoffs, dates):
    """For each date: pick tickers, hold 6 days, record the return. NaN = too few passed."""
    out = []

    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)

        picks = select_growth_stocks(i, passed)

        if len(picks) < MIN_TICKERS:
            out.append(np.nan)
            continue

    
        try:
             weights = final_growth_weights(picks, i)
        except ValueError:
            out.append(np.nan)
            continue
            
        weekly_return = (fwd_returns.loc[i, picks] * weights).sum()
        out.append(weekly_return)

    return pd.Series(out, index=dates)

# counts how many stocks pass the indicator filters on each date
def count_passing(tables, rules, cutoffs, dates):
    """How many tickers passed the rules on each date."""
    counts = []
    for i in dates:
        counts.append(passes(tables, rules, cutoffs, i).sum())
    return pd.Series(counts, index=dates)

# Scores the strategy based on returns, with penalties when too few stocks pass
def score_high_return(cutoffs, dates):
    """ONE number for these cutoffs: average weekly return, with penalties for failed weeks."""
    weekly = weekly_high_return(cutoffs, dates)
    n = count_passing(high_return_tables, HIGH_RET_RULES, cutoffs, dates)

    shortfall = (MIN_TICKERS - n).clip(lower=0)

    weekly = weekly.fillna(-0.10)

    weekly = weekly - 0.01 * shortfall

    return weekly.mean()

In [26]:
result_high_returns = gp_minimize(

    lambda p: -score_high_return(p, tune_dates),
    hr_space, 
    n_calls=30,
    n_initial_points=10,
    acq_func="EI", 
    random_state=0,
    verbose=True
)

best_hr = result_high_returns.x 

# no cutoff comparison 
no_cut_hr = [-np.inf if d == ">" else np.inf for d in HIGH_RET_RULES.values()]


print("best cutoffs:", dict(zip(HIGH_RET_RULES, [round(float(x), 2) for x in best_hr])))
print()


print("                tune      val")
print("tuned:      ", round(score_high_return(best_hr, tune_dates), 4),
      "  ", round(score_high_return(best_hr, validation_dates), 4))
print("no cutoffs: ", round(score_high_return(no_cut_hr, tune_dates), 4),
      "  ", round(score_high_return(no_cut_hr, validation_dates), 4))
print()


tuned_weeks = weekly_high_return(best_hr, validation_dates)
base_weeks = weekly_high_return(no_cut_hr, validation_dates)

wins = (tuned_weeks > base_weeks).sum()
print(wins, "wins out of", len(validation_dates), "validation weeks")


print("tickers passing each validation week:",
      count_passing(high_return_tables, HIGH_RET_RULES, best_hr, validation_dates).tolist())

Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.4420
Function value obtained: 0.0919
Current minimum: 0.0919
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.3461
Function value obtained: 0.0706
Current minimum: 0.0706
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.2752
Function value obtained: 0.0694
Current minimum: 0.0694
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.3806
Function value obtained: 0.0695
Current minimum: 0.0694
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.3496
Function value obtained: 0.0826
Current minimum: 0.0694
Iteration No: 6 started. Evaluating

In [27]:

# Test the growth scores on the most recent data
latest = high_return_tables["adx 3M"].dropna(how="all").index[-1]

passed = passes(high_return_tables, HIGH_RET_RULES, best_hr, latest)

growth_picks = select_growth_stocks(latest, passed)

scores = growth_scores(growth_picks, latest)

print(scores.sort_values(ascending=False))


ABBV    251.589651
KO      196.582754
LLY     192.502426
PFE     145.953570
T.TO    131.199188
ACN     125.700128
CL      111.063513
MRK     104.185209
UNP      77.633784
MO       72.025486
UPS      54.515173
BLK       1.496416
BAC       0.000000
dtype: float64


In [28]:
def show_passing(cutoffs, dates, show_picks=True):
    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)
        names = list(passed[passed].index)
        print(f"{i}  ({len(names)} passed): {names}")

        if show_picks and len(names) >= MIN_TICKERS:
            picks = select_growth_stocks(i, passed)
            dropped = [t for t in names if t not in picks]
            print(f"     held ({len(picks)}): {list(picks)}")
            if dropped:
                print(f"     passed but cut by top-{MAX_TICKERS}: {dropped}")
        print()
latest = high_return_tables["adx 3M"].dropna(how="all").index[-1]
show_passing(best_hr, [latest])

2026-11-25  (13 passed): ['ABBV', 'ACN', 'BAC', 'BLK', 'CL', 'KO', 'LLY', 'MO', 'MRK', 'PFE', 'T.TO', 'UNP', 'UPS']
     held (13): ['T.TO', 'KO', 'ABBV', 'MRK', 'LLY', 'PFE', 'ACN', 'BAC', 'UNP', 'CL', 'UPS', 'MO', 'BLK']



In [29]:

# Calculate weights for the newly selected stocks
final_weights = final_growth_weights(growth_picks, latest)

# Check stock weights
print("Number of stocks:", len(final_weights))
print("Total weight:", round(final_weights.sum() * 100, 2), "%")
print("Largest stock:", round(final_weights.max() * 100, 2), "%")

# Check sector weights
growth_portfolio = pd.DataFrame({
    "Ticker": final_weights.index,
    "Sector": [stock_sectors[t] for t in final_weights.index],
    "Weight": final_weights.values
})

print()
print(calculate_sector_weights(growth_portfolio))
print("Sector limits passed:", check_sector_limit(growth_portfolio))


Number of stocks: 13
Total weight: 100.0 %
Largest stock: 12.44 %

Sector
Communication Services    0.083256
Consumer Defensive        0.245014
Financial Services        0.077434
Healthcare                0.390874
Industrials               0.122042
Technology                0.081379
Name: Weight, dtype: float64
Sector limits passed: True


### Client A — Financial justification


**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

Our high return portfolio strategy mainly focuses on four indicators: ADX, DI spread, beta, and market capitalization. ADX (average directional index) and DI (directional indicator) are used together to measure trend strength and direction. ADX measures the strength of a stock's price trend, while DI spread helps determine whether upward or downward price movement is stronger. Using both 3-month and 1-month periods helps us evaluate longer-term trends while also considering more recent changes in stock momentum.

We also considered beta and market capitalization. Beta measures how sensitive a stock is to movements in the overall market. A higher beta generally means greater exposure to market  movements, which could provide higher returns when the market is performing well, but also creates additional risk. We included market capitalization because smaller companies can sometimes have greater growth potential, although they may also experience greater price fluctuations.

To determine our stock selection thresholds, we used Gaussian Process optimization through the skopt library. This method tests different combinations of indicator thresholds using historical portfolio returns. Instead of testing every possible combination, it uses results from earlier evaluations to guide which combinations to test next. Our HIGH_RET_RULES dictionary specifies which indicators should be above or below the thresholds being optimized. The goal was to find thresholds that produced stronger average six-trading-day portfolio returns in our historical backtest. We separated the historical testing dates into tuning and validation periods so we could evaluate how the selected thresholds performed on data that was not used directly for optimization. If fewer than 10 stocks passed the filters, we used a backup selection method to add eligible stocks until we reached the minimum, prioritizing stocks with higher 3-month ADX values.
 
In terms of the weighting, we decided to use a combination of 50% equal weighting and 50% growth-based weighting. We started by distributing half of the portfolio equally among all selected stocks, ensuring that every stock received a base allocation. We then distributed the remaining half based on each stock's growth score, which we calculated by multiplying its 3-month ADX by its positive 3-month DI spread. Stocks with stronger positive trends received a larger portion of this remaining allocation.

A stock with a negative DI spread received a growth score of zero, meaning it would only receive its base allocation before any adjustments for portfolio limits.

The reason we chose this approach is because we wanted to prioritize stocks showing stronger growth signals without fully relying on those signals. Using equal weighting for half of the portfolio helps maintain diversification and reduce risk of putting too much into only a few stocks. Simultaneously, the growth based portion allows us to take on additional risk in pursuit of a higher return, fitting client A's objective.

Lastly, we capped each individual stock at 15% of the portfolio and limited each sector to a maximum of 40% of the total portfolio value to follow the assignment's requirements. This also helps reduce concentration risk and maintain diversification.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

We compared our portfolio to an equal weighted portfolio holding the exact same stocks, from October 1, 2025 to the most recent date in our data. When we ran it on November 26, 2026, our portfolio returned 30.79% compared to 28.95% for equal weighting. This means our weighting added about 1.84 percentage points more than the equal weighted portfolio. This suggests that allocating more money to stocks with stronger growth scores improved historical returns over the period examined. Our standard deviation of daily returns was slightly higher, 0.967% compared to 0.929% for the equal weighted portfolio, which illustrates the risk to return tradeoff. Our weighting strategy produced higher historical returns but also greater fluctuations in daily performance.

Since Client A prioritizes growth, accepting this additional volatility may be reasonable. However, past performance does not guarantee future returns.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

One situation where this strategy could perform poorly is during a market downturn, especially if most stocks have shown negative momentum over the past three months. Terrible in the sense that there is a clear downwards market shift. Since skopt is run on the parameters for a ticker filter on indictators rather than weighting. Since our portfolio requires at least 10 stocks, we added a backup selection system that chooses from the remaining eligible stocks if fewer than 10 pass the optimized filters. This means that if essentially all tickers perform badly in a three-month window, the optimization may select unreasonably lenient thresholds. For example, in testing, the "optimal" DI spread for 1M and 3M window was negative. This is concerning because a negative DI spread indicates stronger downward directional movement, which goes against our goal of selecting stocks with strong upward momentum. However, our weighting system only rewards positive DI spread, so stocks with negative DI spread receive only their base allocation before any weight adjustments. Also, the program has NO CHOICE but to let them through because of the 10 ticker minimum. And so yeah if the data that is passed through the strategy at a certain moment, just so happens to have very poorly performing general market trends, then the program will let in a bunch of tickers out of desparity for fufilling the minimum ticker requirement of the portfolio, and thus the momentum indicators don't actually function as they're supposed to in the sense that they should be positive. 

Regarding beta and market cap, beta is pulled from yfinance that has no historical beta values so that value is pulled from now. Also, market cap is calculated using SharesOutstanding and close price data. Beta uses currently available data and market capitalization may rely on currently reported shares outstanding, our historical backtest could contain look-ahead bias. Basically the beta and at least the share componenet of market cap are not a reflection of the actual ticker data in the tuning and validation windows. And so it isn't an accurate representation of beta and market cap in that moment. There is also a risk of overfitting, since the Gaussian Process optimizes thresholds based on historical performance that may not repeat in the future.

Although our stock and sector limits help reduce concentration risk, they cannot completely protect the portfolio from market-wide declines.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

In [30]:
# CONSTANTS
lv_window = 63
MAX_TICKERS = 20 #same variables just restating for clarity
MIN_TICKERS = 10 #same variables just restating for clarity
sector_cap = 0.4
max_per_sector = 8  

min_tickers = MIN_TICKERS  #this was done in case i wanted to set a higher minimum to force the greedy selection to have more than 
# 10 tickers as that was the norm in testing. 

sector_list = [
    "Utilities",
    "Consumer Defensive",
    "Healthcare",
    "Real Estate",
    "Financial Services",
    "Industrials",
    "Energy",
    "Basic Materials",
    "Communication Services",
    "Consumer Cyclical",
    "Technology"
]

In [31]:
def pull_stock_sectors (tickers):
    sectors = []

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        sector = stock.info.get("sector")

        sectors.append({
            "Ticker": ticker,
            "Sector" :sector
        })
    return pd.DataFrame(sectors)


sectors = pull_stock_sectors(stock_data.keys())
print(sectors)

     Ticker                  Sector
0      AAPL              Technology
1      ABBV              Healthcare
2       ABT              Healthcare
3       ACN              Technology
4       AIG      Financial Services
5      AMZN       Consumer Cyclical
6       AXP      Financial Services
7        BA             Industrials
8       BAC      Financial Services
9     BB.TO              Technology
10     BIIB              Healthcare
11      BLK      Financial Services
12      BMY              Healthcare
13        C      Financial Services
14      CAT             Industrials
15       CL      Consumer Defensive
16       KO      Consumer Defensive
17      LLY              Healthcare
18      LMT             Industrials
19       MO      Consumer Defensive
20      MRK              Healthcare
21      PEP      Consumer Defensive
22      PFE              Healthcare
23    RY.TO      Financial Services
24       PM      Consumer Defensive
25     PYPL      Financial Services
26     QCOM              Tec

In [32]:
def lookback_returns(date, window=lv_window):
    window_returns = returns.loc[:date].tail(window)
    return window_returns

def portfolio_vol(window_returns, tickers):
    port_returns = window_returns[tickers].mean(axis=1)
    return port_returns.std()

sector_of = sectors.set_index("Ticker")["Sector"].to_dict()

def get_sector(ticker):
    s = sector_of.get(ticker)
    if s is None:
        return (f"Unknown_{ticker}")
    return s


def sector_counts(tickers):
    counts = {}
    for t in tickers:
        s = get_sector(t)
        counts[s] = counts.get(s, 0) + 1
    return counts

def over_cap_sectors(selected):
    if len(selected) == 0:
        return set()
    over = set()
    for s, c in sector_counts(selected).items():
        if c / len(selected) > sector_cap + 1e-9:
            over.add(s)
    return over

def usable_universe(window_returns, min_coverage=0.9):
    coverage = window_returns.notna().mean()
    return list(coverage[coverage >= min_coverage].index)

def eligible_candidates(selected, universe):
    counts = sector_counts(selected)
    over = over_cap_sectors(selected)

    base = []
    for t in universe:
        if t not in selected and counts.get(get_sector(t), 0) < max_per_sector:
            base.append(t)

    preferred = [t for t in base if get_sector(t) not in over]

    if len(preferred) > 0:
        return preferred
    return base 

In [33]:
def build_low_vol(date, universe=None, verbose=False, min_n= min_tickers, max_n=MAX_TICKERS):
    window = lookback_returns(date)

    if universe is None:
        universe = usable_universe(window)

    selected = []
    current_vol = np.inf 

    while len(selected) < max_n:
        candidates = eligible_candidates(selected, universe)
        if len(candidates) == 0:
            break

        best_ticker = None
        best_vol = np.inf
        for i in candidates:
            v = portfolio_vol(window, selected + [i])
            if pd.notna(v) and v < best_vol:
                best_vol = v
                best_ticker = i

        if best_ticker is None:
            break

        # STOPPING RULE: all three must be true to stop. subject to change depending on tests(don't forget leon)
        raises_vol = best_vol >= current_vol
        sectors_ok = len(selected) > 0 and len(over_cap_sectors(selected)) == 0
        big_enough = len(selected) >= min_n

        if raises_vol and sectors_ok and big_enough:
            if verbose:
                print(f"STOP: adding {best_ticker} would raise vol to {best_vol:.5f}")
            break

        selected.append(best_ticker)
        current_vol = best_vol

        if verbose:
            print(f"N={len(selected):2d}  added {best_ticker:8s} ({get_sector(best_ticker)})  vol={current_vol:.5f}")

    # warnings 
    if len(over_cap_sectors(selected)) > 0:
        print(f"WARNING {date}: sectors over cap: {over_cap_sectors(selected)}")
    if len(selected) < min_n:
        print(f"WARNING {date}: only {len(selected)} tickers selected (min {min_n})")

    return selected

In [34]:
def min_var_weights(tickers, window_returns, lo=0.05, hi=0.15, sector_max=0.40):
    R = window_returns[tickers].dropna()
    cov = R.cov().values * 1000000
    n = len(tickers)

    # weighting must equal 1 :)
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]

    # Sector condition
    for s in set(get_sector(t) for t in tickers):
        idx = [k for k, t in enumerate(tickers) if get_sector(t) == s]
        cons.append({"type": "ineq", "fun": lambda w, idx=idx: sector_max - w[idx].sum()})

    # Minimizing variance 
    res = minimize(lambda w: w @ cov @ w, np.ones(n) / n,
                   bounds=[(lo, hi)] * n, constraints=cons, method="SLSQP")

    if not res.success:
        raise ValueError("Min-variance optimization failed: " + res.message)

    return pd.Series(res.x, index=tickers)

In [35]:

def make_low_vol_portfolio(selected, date):

    n = len(selected)
    if n == 0:
        print("ERROR: empty portfolio")
        return pd.DataFrame(columns=["Ticker", "Sector", "Weight"])

    weights = min_var_weights(selected, lookback_returns(date))

    portfolio = pd.DataFrame({
        "Ticker": selected,
        "Sector": [get_sector(t) for t in selected],
        "Weight": weights.values,
    })

    problems = []
    if not (min_tickers <= n <= MAX_TICKERS):
        problems.append(f"N={n} outside [{min_tickers}, {MAX_TICKERS}]")
    if abs(portfolio["Weight"].sum() - 1) > 1e-9:
        problems.append(f"weights sum to {portfolio['Weight'].sum()}")
    if portfolio["Weight"].min() < 0.05 - 1e-9 or portfolio["Weight"].max() > 0.15 + 1e-9:
        problems.append("a weight is outside 5%-15%")
    if not check_sector_limit(portfolio):
        problems.append(f"sector over 40%: {calculate_sector_weights(portfolio).to_dict()}")

    if problems:
        print("PORTFOLIO CHECK FAILED:")
        for p in problems:
            print("   -", p)
    else:
        print(f"All checks passed ({n} tickers, weights {portfolio['Weight'].min():.2%} to {portfolio['Weight'].max():.2%})")

    return portfolio

In [36]:
latest = returns.dropna(how="all").index[-1]
known_sector = [t for t in usable_universe(lookback_returns(latest)) if stock_sectors.get(t)]
lv_picks = build_low_vol(latest, universe=known_sector, verbose=True)
lv_portfolio = make_low_vol_portfolio(lv_picks, latest)
print(lv_portfolio)
print(calculate_sector_weights(lv_portfolio))

N= 1  added RY.TO    (Financial Services)  vol=0.01151
N= 2  added KO       (Consumer Defensive)  vol=0.00870
N= 3  added PFE      (Healthcare)  vol=0.00880
N= 4  added TD.TO    (Financial Services)  vol=0.00881
N= 5  added MO       (Consumer Defensive)  vol=0.00861
N= 6  added AMZN     (Consumer Cyclical)  vol=0.00830
N= 7  added T.TO     (Communication Services)  vol=0.00749
N= 8  added LMT      (Industrials)  vol=0.00725
N= 9  added CAT      (Industrials)  vol=0.00723
N=10  added PM       (Consumer Defensive)  vol=0.00718
STOP: adding AIG would raise vol to 0.00722
All checks passed (10 tickers, weights 5.00% to 15.00%)
  Ticker                  Sector    Weight
0  RY.TO      Financial Services  0.109513
1     KO      Consumer Defensive  0.150000
2    PFE              Healthcare  0.050000
3  TD.TO      Financial Services  0.050000
4     MO      Consumer Defensive  0.150000
5   AMZN       Consumer Cyclical  0.125127
6   T.TO  Communication Services  0.148336
7    LMT             Indu

In [37]:
window = lookback_returns(latest)
w = lv_portfolio.set_index("Ticker")["Weight"]
mv_vol = (window[w.index] * w).sum(axis=1).std()
eq_vol = window[w.index].mean(axis=1).std()
print(f"Min-variance daily vol: {mv_vol:.5f}   Equal-weight daily vol: {eq_vol:.5f}")

Min-variance daily vol: 0.00668   Equal-weight daily vol: 0.00718


### Client B — Financial justification


**Why does this portfolio make financial sense?** 

The main idea behind this portfolio is that it runs on a greedy system where it will only add a ticker to the portfolio if the addition of the ticker reduces the volatility of the portfolio. That volatility is calculated from the mean standard deviation over the past 63 days of the returns of each ticker (pct_change()), which is equivilant to 3 trading months. The greedy system is designed such that it will meet the minimum 10 ticker portfolio requirement. After the portfolio is created, using scipy skopt minimize, the weightings are determined as such that the tickers within that portfolio will form the lowest possible volatility. Again that volatility is calculated the same as before. Moreover, the process heeds ticker weighting requirement for individual tickers, 15% max, and sectors 40%. 

This portfolio makes financial sense as we stripped the tickers, the portfolio composition, and ticker weighting into its rawest form and optimized that. The main goal of this portfolio is to minimize volatility. And so all measures of the portfolio composition were directed by the effect on the total portfolio volatility. The greedy system will ensure that all tickers added aren't increasing the equal weighted portfolio's volatility, and that every ticker has to fight for its position on the portfolio. The skopt minimize will also make every ticker fight for its weighting in the portfolio by only giving weighting to those who result in the portfolio as a whole getting the lowest volatility. 

Moreover, the period of consideration for volatility, 3 months of trading days, is a fair measurement not only for the 6 day trading window of the competition but also for a portfolio in a "long-term" sense. The benefits of the 3 month window is that any short term price volatility caused by macroeconomic influence (for example a trump truth social post or smth) gets smoothed out such that something that isn't correlated with the ticker itself doesn't determine its overall volatility. Futhermore, past the 3 month window, theoretically the smoothing out would now smooth out periods of volatility that are sort of important for our consideration of admission into the portfolio. In like a 6 month or 1 year period, a very shaky volatile 1 month period of the stocks day to day returns could have been smoothed out by the mean and thus wouldn't fully reflect on the tickers volatility in a near term (6d trading window) and future (6 month to 1 year) volatility levels. 

We did consider things like beta and ATR% (both things are already calculated as indicators for the high return portfolio). However it was concluded that beta and ATR% were essentially measuring the exact same thing as the aformentioned volatility measurement. They were all measurements of a stocks price movement, and so having 3 of them would be no different than having 1, minus all the noise and uncertainty involved in weighting them on impact. Moreover in the case of beta, we felt that since beta was something that we called from yfinance, it didn't make sense to include it at least for low volatility when we are trying to be as precise as possible, when we couldn't calculate it ourselves. Obviously, I could write a program to calculate beta, but it was just inefficient, and simply unneccesary given the fact that our actual method of volatility calculation did the job. 


**What evidence supports your strategy?** 

The evidence that supports this strategy mainly just come from testing. I initially did a strategy where I ranked indicators based of ATR%, beta, and rolling price std. Then I took the average ranking of each ticker across indicators, and took the lowest 20. Equal weight distribution with the idea that max tickers with the lowest possible weighting for each would definetely minimize the portfolio volatility! Obviously not. And well as I continued to look into different ideas, ultimately it was this one that had the lowest portfolio volatility. And the results spoke quite strongly to me and I decided that was pretty evident that what I was doing was logical and those previous failures were evidence to the "strength" of my current one. 

Moreover, the theory that I went over in the previous question similarily applies here, where I felt that my strategy was theoretically coherent. The mean std of returns is a measurement of volatility. Having it be greedy is a solid way of reducing unneccesary tickers in a portfolio. And having skopt minimize to determine weighting for tickers does work in comparison to equal weighting. All those things are true, and support my "trial and error" process of determining a strategy. And all together, I believe that stands as reasonable evidence to my strategy. 

**What could go wrong?** 

I believe the portfolio does a great job at generally diversifing holdings. You won't see a large grouping of tickers in the same sector simply because the greedy system will cut that before it even gets a chance. I would argue the portfolio could defend well against sector risk and sector specific price trends (for example, tech this past month). But for limitations, it is honestly the scope in how the volatility of tickers is measured. There were plans to use eps (earnings per share), roe (return on equity), dividends, etc. Unfortunately we chose not to pursue those indicators since we felt they were less applicable to the 6d trading competition window (although they are very applicable to the volatility of tickers in a long term window). The fact that those indicators aren't taken into consideration, rather we use purely indicators pulled from raw pricing data, the portfolio is at risk of tickers that hypothetically just had a super chill past 3 months, but EVERY other signal points to it being a roller coaster in the future. Ontop of that, we did not look into how exchange rates could affect stuff, so tickers in the portfolio could be american and thus be susceptible to price volatility purely off the volatility of the USD/CAD exchange rate. 

Those were all limitations that we were fully aware of, but purely for the sake of having an operable portfolio that performed the best in that 6d trading window but still made logical coherent sense in a longer time frame, we chose not to consider them into our strategy. 



## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [38]:
start_date = close.index[close.index <= latest][-252]   # last ~1 year of trading days before the portfolio date

def portfolio_value(tickers, weights):

    price_dict = {}
    for i in range(len(tickers)):
        price_dict[tickers[i]] = stock_data[tickers[i]]["Close_CAD"]

    prices = pd.DataFrame(price_dict).sort_index()

    prices = prices.ffill()                  
    prices = prices.loc[start_date:]         
    prices = prices.dropna()

    value = 0
    for i in range(len(tickers)):
        shares = weights[i] / prices[tickers[i]].iloc[0]
        value = value + shares * prices[tickers[i]]

    return value

def compare(tickers, our_weight):

    equal_weight = []
    for i in range(len(tickers)):
        equal_weight.append(1 / len(tickers))

    ours = portfolio_value(tickers, our_weight)
    equal = portfolio_value(tickers, equal_weight)

    table = pd.DataFrame()
    table["Portfolio"] = ["Our weights", "Equal weights"]
    table["Return (%)"] = [(ours.iloc[-1] / ours.iloc[0] - 1) * 100,
                           (equal.iloc[-1] / equal.iloc[0] - 1) * 100]
    table["Daily volatility (%)"] = [ours.pct_change().std() * 100,
                                     equal.pct_change().std() * 100]

    print("Period:", ours.index[0], "->", ours.index[-1])

    return table.round(5)

#client A: tickers and weights from the growth strategy
tickers_A = list(final_weights.index)
weights_A = list(final_weights.values)

#client B: tickers and weights from the low volatility strategy
tickers_B = list(lv_portfolio["Ticker"])
weights_B = list(lv_portfolio["Weight"])

print("Client A")
display(compare(tickers_A, weights_A))

print("Client B")
display(compare(tickers_B, weights_B))

Client A
Period: 2025-12-02 -> 2026-11-25


,Portfolio,Return (%),Daily volatility (%)
0,Our weights,22.35211,0.97036
1,Equal weights,20.08071,0.92873


Client B
Period: 2025-12-02 -> 2026-11-25


,Portfolio,Return (%),Daily volatility (%)
0,Our weights,17.73845,0.69448
1,Equal weights,19.73165,0.72370


## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [39]:
budget_A = 1_000_000
budget_B = 750_000
GROUP_NUMBER = "04"
TOL = 1e-6

fx = cadusd.iloc[-1]   # USD per 1 CAD on the purchase date

def fees_cad(shares):
    return np.minimum(2.15, 0.001 * shares) / fx

def build_portfolio(weights, budget):
    weights = weights / weights.sum()
    tickers = list(weights.index)
    price_cad = pd.Series({t: stock_data[t]["Close_CAD"].iloc[-1] for t in tickers})

    # setting aside the max 2.15 fee :)
    invest = budget - len(tickers) * 2.15 / fx

    shares = invest * weights / price_cad
    value = shares * price_cad

    return pd.DataFrame({
        "Ticker":   tickers,
        "Price":    [stock_data[t]["Close"].iloc[-1] for t in tickers],
        "Currency": [stock_info[t]["currency"] for t in tickers],
        "Shares":   shares.values,
        "Value":    value.values, 
        "Weight":   (value / value.sum()).values,
    }, index=range(1, len(tickers) + 1))

Portfolio_Final_A = build_portfolio(final_weights, budget_A)
Portfolio_Final_B = build_portfolio(lv_portfolio.set_index("Ticker")["Weight"], budget_B)

def validate_portfolio(port, budget):
    sectors = port["Ticker"].map(stock_sectors)
    spent = port["Value"].sum() + fees_cad(port["Shares"]).sum()

    checks = {
        "Valid and eligible tickers": port["Ticker"].isin(eligible).all() and port["Ticker"].is_unique,
        "10-20 stocks":               MIN_TICKERS <= len(port) <= MAX_TICKERS,
        "No stock over 15%":          port["Weight"].max() <= 0.15 + TOL,
        "No sector over 40%":         sectors.fillna("").ne("").all()
                                      and port.groupby(sectors)["Weight"].sum().max() <= sector_cap + TOL,
        "Weights sum to 100%":        abs(port["Weight"].sum() - 1) <= TOL,
        "Purchases + fees <= budget": spent <= budget + TOL,
    }
    return pd.DataFrame({"Result": ["PASS" if ok else "FAIL" for ok in checks.values()]},
                        index=list(checks.keys()))

Validation_A = validate_portfolio(Portfolio_Final_A, budget_A)
Validation_B = validate_portfolio(Portfolio_Final_B, budget_B)

display(pd.concat({"Client A": Validation_A["Result"],
                   "Client B": Validation_B["Result"]}, axis=1))

,Client A,Client B
Valid and eligible tickers,PASS,PASS
10-20 stocks,PASS,PASS
No stock over 15%,PASS,PASS
No sector over 40%,PASS,PASS
Weights sum to 100%,PASS,PASS
Purchases + fees <= budget,PASS,PASS


## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [40]:
for client, port, budget in [("A", Portfolio_Final_A, budget_A), ("B", Portfolio_Final_B, budget_B)]:
    purchases = port["Value"].sum()
    fees = fees_cad(port["Shares"]).sum()

    print(f"CLIENT {client}")
    display(port.style.format({"Price": "{:,.2f}", "Shares": "{:,.6f}",
                               "Value": "${:,.2f}", "Weight": "{:.2%}"}))
    display(pd.DataFrame({"Amount (CAD)": [budget, purchases, fees, purchases + fees,
                                           budget - purchases - fees]},
                         index=["Initial budget", "Stock purchases", "Transaction fees",
                                "Total spent", "Remaining cash"]).style.format("${:,.2f}"))

    port[["Ticker", "Shares"]].to_csv(f"Stocks_Group_{GROUP_NUMBER}_{client}.csv", index=False)

CLIENT A


,Ticker,Price,Currency,Shares,Value,Weight
1,T.TO,15.31,CAD,"5,437.818065","$83,253.00",8.33%
2,KO,97.59,USD,757.960547,"$105,575.74",10.56%
3,ABBV,294.48,USD,295.867906,"$124,355.76",12.44%
4,MRK,140.29,USD,369.716981,"$74,030.10",7.40%
5,LLY,"1,392.22",USD,52.429459,"$104,182.67",10.42%
6,PFE,28.46,USD,"2,173.531762","$88,290.32",8.83%
7,ACN,240.49,USD,237.074413,"$81,375.55",8.14%
8,BAC,48.54,USD,555.133353,"$38,460.00",3.85%
9,UNP,298.38,USD,152.545111,"$64,965.12",6.50%
10,CL,99.66,USD,536.954668,"$76,378.43",7.64%


,Amount (CAD)
Initial budget,"$1,000,000.00"
Stock purchases,"$999,960.11"
Transaction fees,$11.73
Total spent,"$999,971.84"
Remaining cash,$28.16


CLIENT B


,Ticker,Price,Currency,Shares,Value,Weight
1,RY.TO,269.26,CAD,305.026447,"$82,131.42",10.95%
2,KO,97.59,USD,807.638894,"$112,495.40",15.00%
3,PFE,28.46,USD,923.137494,"$37,498.47",5.00%
4,TD.TO,164.71,CAD,227.663555,"$37,498.47",5.00%
5,MO,76.28,USD,"1,033.265311","$112,495.40",15.00%
6,AMZN,244.43,USD,268.985267,"$93,841.56",12.51%
7,T.TO,15.31,CAD,"7,266.318208","$111,247.33",14.83%
8,LMT,599.24,USD,47.807871,"$40,889.56",5.45%
9,CAT,543.69,USD,95.467843,"$74,083.36",9.88%
10,PM,184.32,USD,181.650807,"$47,788.35",6.37%


,Amount (CAD)
Initial budget,"$750,000.00"
Stock purchases,"$749,969.31"
Transaction fees,$8.62
Total spent,"$749,977.94"
Remaining cash,$22.06


### Contribution and AI Declaration ###
**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.